In [2]:
import os
import time
import json
from pathlib import Path
import pandas as pd
import glob
import numpy as np
import pprint
import requests
from datetime import datetime

In [3]:
df = pd.read_csv("vote_outcomes_2014-2026.csv")


In [5]:
df.sort_values("start_date", inplace=True)

In [7]:
df.drop(columns="absolute_majority", inplace=True)

In [9]:
df

,Unnamed: 0,vote_id,decision_id,start_date,method,outcome,attendees,votes_favor,votes_against,heading,master_doc,item_number,abstentions
0,0,eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975331,eli/dl/event/MTG-PL-2025-10-23-DEC-180520,2025-10-23T13:08:06+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,571.0,220.0,341.0,Proposal to reject the Council position,NaN,NaN,NaN
19,19,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195775,2026-07-09T13:21:39+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,607.0,314.0,276.0,Proposal to reject the Council position,NaN,NaN,NaN
29,29,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195776,2026-07-09T13:22:14+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,605.0,117.0,422.0,Draft legislative act,NaN,NaN,NaN
24,24,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195777,2026-07-09T13:22:31+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,596.0,114.0,422.0,Draft legislative act,NaN,NaN,NaN
33,33,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195778,2026-07-09T13:22:50+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,611.0,369.0,236.0,Draft legislative act,NaN,NaN,NaN
13,13,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195779,2026-07-09T13:23:17+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,606.0,161.0,433.0,Draft legislative act,NaN,NaN,NaN
16,16,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195780,2026-07-09T13:23:32+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,605.0,120.0,433.0,Draft legislative act,NaN,NaN,NaN
25,25,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195781,2026-07-09T13:23:46+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,602.0,158.0,433.0,Draft legislative act,NaN,NaN,NaN
34,34,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195782,2026-07-09T13:24:03+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,604.0,119.0,434.0,Draft legislative act,NaN,NaN,NaN
14,14,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195783,2026-07-09T13:24:23+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,600.0,345.0,237.0,Draft legislative act,NaN,NaN,NaN


In [10]:
#Complete the dataframe with the vote outcome of the ones with Nan
decisions_folder = "decisions_json_dumps"

print("Scanning for missing outcomes...")

# 1. Filter for rows where 'outcome' is missing (NaN)
missing_outcome_mask = df['outcome'].isna()

# 2. Iterate through only the rows that need fixing
for index, row in df[missing_outcome_mask].iterrows():
    
    # Safely get the decision_id (e.g., 'eli/dl/event/MTG-PL-2025-10-23-DEC-180605')
    decision_id_full = str(row.get('decision_id', ''))
    
    # Extract the file name part (everything after the last '/')
    file_base_name = decision_id_full.split("/")[-1]
    file_path = os.path.join(decisions_folder, f"{file_base_name}.json")
    
    # 3. Open the file if it exists
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            decision_data = json.load(f)
            
            # 4. Navigate safely into the JSON data array
            if "data" in decision_data and len(decision_data["data"]) > 0:
                target_dict = decision_data["data"][0]
                
                # Extract the English reference text
                ref_text_en = target_dict.get("referenceText", {}).get("en", None)
                
                if ref_text_en:
                    # Inject it into the outcome column
                    df.at[index, 'outcome'] = ref_text_en
                    
                    # BONUS: Auto-fill the method and Council Win column to keep your data perfectly clean!
                    if "without vote" in ref_text_en.lower():
                        df.at[index, 'method'] = "Auto-adopted"
                        df.at[index, 'Council win or loss'] = "Council win"
                        
                    print(f"Patched {file_base_name} -> {ref_text_en}")
    else:
        print(f"Warning: File not found for {file_base_name}")

print("\n--- PATCHING COMPLETE ---")

# Let's peek at the fixed rows to verify!
fixed_rows = df[df['decision_id'].str.contains('DEC-180605|DEC-181469', na=False)]
print(fixed_rows[['decision_id', 'method', 'outcome', 'Council win or loss']])

Scanning for missing outcomes...
Patched MTG-PL-2025-10-23-DEC-180605 -> Approval without vote
Patched MTG-PL-2025-11-13-DEC-181469 -> Approval without vote
Patched MTG-PL-2025-11-13-DEC-181470 -> Approval without vote
Patched MTG-PL-2026-01-22-DEC-184076 -> Approval without vote
Patched MTG-PL-2026-03-26-DEC-189897 -> Approval without vote
Patched MTG-PL-2026-03-26-DEC-189900 -> Approval without vote
Patched MTG-PL-2026-03-26-DEC-189899 -> Approval without vote
Patched MTG-PL-2026-03-26-DEC-189898 -> Approval without vote

--- PATCHING COMPLETE ---
                                 decision_id        method  \
1  eli/dl/event/MTG-PL-2025-10-23-DEC-180605  Auto-adopted   
2  eli/dl/event/MTG-PL-2025-11-13-DEC-181469  Auto-adopted   

                 outcome Council win or loss  
1  Approval without vote         Council win  
2  Approval without vote         Council win  


In [15]:
fixed_rows

,Unnamed: 0,vote_id,decision_id,start_date,method,outcome,attendees,votes_favor,votes_against,heading,master_doc,item_number,abstentions,Council win or loss
1,1,eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975337,eli/dl/event/MTG-PL-2025-10-23-DEC-180605,NaN,Auto-adopted,Approval without vote,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Council win
2,2,eli/dl/event/MTG-PL-2025-11-13-VOT-ITM-975467,eli/dl/event/MTG-PL-2025-11-13-DEC-181469,NaN,Auto-adopted,Approval without vote,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Council win


In [16]:
df

,Unnamed: 0,vote_id,decision_id,start_date,method,outcome,attendees,votes_favor,votes_against,heading,master_doc,item_number,abstentions,Council win or loss
0,0,eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975331,eli/dl/event/MTG-PL-2025-10-23-DEC-180520,2025-10-23T13:08:06+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,571.0,220.0,341.0,Proposal to reject the Council position,NaN,NaN,NaN,NaN
19,19,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195775,2026-07-09T13:21:39+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,607.0,314.0,276.0,Proposal to reject the Council position,NaN,NaN,NaN,NaN
29,29,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195776,2026-07-09T13:22:14+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,605.0,117.0,422.0,Draft legislative act,NaN,NaN,NaN,NaN
24,24,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195777,2026-07-09T13:22:31+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,596.0,114.0,422.0,Draft legislative act,NaN,NaN,NaN,NaN
33,33,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195778,2026-07-09T13:22:50+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,611.0,369.0,236.0,Draft legislative act,NaN,NaN,NaN,NaN
13,13,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195779,2026-07-09T13:23:17+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,606.0,161.0,433.0,Draft legislative act,NaN,NaN,NaN,NaN
16,16,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195780,2026-07-09T13:23:32+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,605.0,120.0,433.0,Draft legislative act,NaN,NaN,NaN,NaN
25,25,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195781,2026-07-09T13:23:46+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,602.0,158.0,433.0,Draft legislative act,NaN,NaN,NaN,NaN
34,34,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195782,2026-07-09T13:24:03+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,604.0,119.0,434.0,Draft legislative act,NaN,NaN,NaN,NaN
14,14,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195783,2026-07-09T13:24:23+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,600.0,345.0,237.0,Draft legislative act,NaN,NaN,NaN,NaN


In [18]:
#create separate column for deciding if the eu council won or not
def determine_winner(row):
    # Safely grab the outcome as an uppercase string
    outcome = str(row['outcome']).strip().upper()
    
    # 1. Explicit Approvals
    # This catches "Approval without vote" AND the "Approved" instances 
    # where a roll-call vote to reject the Council failed.
    if outcome in ["APPROVED", "APPROVAL WITHOUT VOTE"]:
        return "Council win"
        
    # 2. Modern API Labels & Historical +/- Parsing
    if "REJECTED" in outcome:
        # Parliament's motion to reject/amend failed
        return "Council win"
        
    if "ADOPTED" in outcome:
        # Parliament's motion to reject/amend succeeded
        return "Council loss"
        
    # 3. Mathematical Fail-Safe (based on your exact observation)
    # If the outcome text is ever missing or ambiguous, we check the raw numbers!
    v_fav = pd.to_numeric(row['votes_favor'], errors='coerce')
    v_agn = pd.to_numeric(row['votes_against'], errors='coerce')
    
    if pd.notna(v_fav) and pd.notna(v_agn):
        if v_agn > v_fav:
            return "Council win"  # The motion against the Council failed
        elif v_fav > v_agn:
            return "Council loss" # The motion against the Council succeeded
            
    return "Unknown"

# Apply the function across all rows (axis=1) in your merged dataframe
df['Council win or loss'] = df.apply(determine_winner, axis=1)

# Let's verify the results, specifically looking at the scenarios you highlighted!
print("=== OUTCOME DISTRIBUTION ===")
print(df['Council win or loss'].value_counts(dropna=False))

print("\n=== VERIFYING YOUR SPECIFIC EDGE CASES ===")
# Show a sample of rows where there WAS a vote, but the Council still won
edge_cases = df[(df['method'].str.contains('Roll Call|ELECTRONIC', na=False, case=False)) & 
                      (df['Council win or loss'] == 'Council win')]

print(edge_cases[['decision_id', 'heading', 'method', 'outcome', 'votes_favor', 'votes_against', 'Council win or loss']].head(10))

=== OUTCOME DISTRIBUTION ===
Council win or loss
Council win     42
Council loss     2
Unknown          1
Name: count, dtype: int64

=== VERIFYING YOUR SPECIFIC EDGE CASES ===
                                  decision_id  \
0   eli/dl/event/MTG-PL-2025-10-23-DEC-180520   
19  eli/dl/event/MTG-PL-2026-07-09-DEC-195775   
29  eli/dl/event/MTG-PL-2026-07-09-DEC-195776   
24  eli/dl/event/MTG-PL-2026-07-09-DEC-195777   
13  eli/dl/event/MTG-PL-2026-07-09-DEC-195779   
16  eli/dl/event/MTG-PL-2026-07-09-DEC-195780   
25  eli/dl/event/MTG-PL-2026-07-09-DEC-195781   
34  eli/dl/event/MTG-PL-2026-07-09-DEC-195782   
14  eli/dl/event/MTG-PL-2026-07-09-DEC-195783   
32  eli/dl/event/MTG-PL-2026-07-09-DEC-195784   

                                    heading  \
0   Proposal to reject the Council position   
19  Proposal to reject the Council position   
29                   Draft legislative act    
24                   Draft legislative act    
13                   Draft legislative act    
16

In [19]:
df

,Unnamed: 0,vote_id,decision_id,start_date,method,outcome,attendees,votes_favor,votes_against,heading,master_doc,item_number,abstentions,Council win or loss
0,0,eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975331,eli/dl/event/MTG-PL-2025-10-23-DEC-180520,2025-10-23T13:08:06+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,571.0,220.0,341.0,Proposal to reject the Council position,NaN,NaN,NaN,Council win
19,19,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195775,2026-07-09T13:21:39+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,607.0,314.0,276.0,Proposal to reject the Council position,NaN,NaN,NaN,Council win
29,29,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195776,2026-07-09T13:22:14+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,605.0,117.0,422.0,Draft legislative act,NaN,NaN,NaN,Council win
24,24,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195777,2026-07-09T13:22:31+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,596.0,114.0,422.0,Draft legislative act,NaN,NaN,NaN,Council win
33,33,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195778,2026-07-09T13:22:50+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,611.0,369.0,236.0,Draft legislative act,NaN,NaN,NaN,Council loss
13,13,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195779,2026-07-09T13:23:17+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,606.0,161.0,433.0,Draft legislative act,NaN,NaN,NaN,Council win
16,16,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195780,2026-07-09T13:23:32+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,605.0,120.0,433.0,Draft legislative act,NaN,NaN,NaN,Council win
25,25,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195781,2026-07-09T13:23:46+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,602.0,158.0,433.0,Draft legislative act,NaN,NaN,NaN,Council win
34,34,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195782,2026-07-09T13:24:03+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,604.0,119.0,434.0,Draft legislative act,NaN,NaN,NaN,Council win
14,14,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195783,2026-07-09T13:24:23+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,600.0,345.0,237.0,Draft legislative act,NaN,NaN,NaN,Council win


In [23]:
df["Council win or loss"].value_counts(normalize=True, dropna=False) * 100
#These are the percentages of the eu council winning the votes


Council win or loss
Council win     93.333333
Council loss     4.444444
Unknown          2.222222
Name: proportion, dtype: float64

In [24]:
df.to_csv("thursday_vote_outcomes_council_win_or_lose.csv")